# Forecast Error Builder (feeds the All Dashboard sigma-based Min)

Reads up to **12 monthly forecast files** (the `FD ...` outputs of the forecast engine) and the **latest pmovdcE** (actual demand), and calculates

`Forecast Error = Actual Demand - Forecast Demand`  per month, per Brc + P/N (and National).

**How the months line up**

| | |
|---|---|
| Forecast file dated *2 Nov 2026* | forecast for **Nov 2026** (`FORECAST_MONTH_OFFSET = 0`) |
| pmovdcE `D-1` | the month before the pmovdcE date (e.g. file of 2 Nov 2026 -> Oct 2026) |
| `F-k`, `E-k` | forecast / error for the same month as `D-k`  (k = 1 is the most recent month) |

A forecast file whose month has no actual yet (e.g. the file of the current month) is skipped automatically.

**Output** (`OUTPUT_FILE`): sheet **Forecast History** (`Brc | P/N | F-1 ... F-12`) is the input of
`FORECAST_HISTORY_FILE` in *All_Dashboard_1_2_6_Claude*; the other sheets are for checking
(errors, sigma, bias, which files were used).

In [ ]:
# ============================================================
# CONFIGURATION -- EDIT HERE
# ============================================================
# Actual demand: the LATEST pmovdcE file (D-1 ... D-12 are the actuals)
ACTUAL_FILE = "pmovdcE 19 Non Steron 11Nov26.xlsx"
ACTUAL_SHEET = "Non Steron"
ACTUAL_SKIPROWS = 4
ACTUAL_D1_MONTH = None        # month of D-1 as "YYYY-MM", e.g. "2026-10".
                              # None = the month before the date written in ACTUAL_FILE's name.

# Forecast files: give a list, or leave the list empty and use a glob pattern.
FORECAST_FILES = [
    # "FD 19 Non Steron Nov25.xlsx",
    # "FD 19 Non Steron Dec25.xlsx",
    # ...
]
FORECAST_FILE_GLOB = "FD 19 Non Steron *.xlsx"       # used when FORECAST_FILES is empty
FORECAST_SHEETS = ["National", "Branch"]         # sheets read from every forecast file
FORECAST_VALUE_COLUMN = "FD_final"               # the forecast that is actually used for planning
                                                 # ("FD_forecast" = the unrounded value)

# Which month does a forecast file forecast?
#   The month written in its file name + FORECAST_MONTH_OFFSET.
#   0 -> a file dated 2 Nov 2026 forecasts Nov 2026 (M-1 of that run).
FORECAST_MONTH_OFFSET = 0
# Per-file override (file name -> "YYYY-MM") if a name has no date or a different meaning.
FORECAST_MONTH_OVERRIDE = {
    # "FD 19 Non Steron Nov25.xlsx": "2025-11",
}

# Sigma method and minimum months (same meaning as in the All Dashboard notebook)
SIGMA_METHOD = "STDEV"        # "STDEV" = sample st.dev. of the error (Excel STDEV) | "RMSE" (also captures bias)
SIGMA_MIN_MONTHS = 6          # fewer months with both actual and forecast -> sigma left blank

# A part (Brc + P/N) that exists in the latest pmovdcE but is NOT in an older forecast file:
#   "zero" -> forecast = 0 for that month (the error then equals the actual demand)
#   "skip" -> left blank, that month is ignored in sigma
# A month that has no forecast file at all always stays blank.
MISSING_FORECAST = "zero"

OUTPUT_FILE = None            # None -> "Forecast History <D-1 month>.xlsx"


In [ ]:
import os, re, glob, datetime
import numpy as np
import pandas as pd

MONTHS = {m: i for i, m in enumerate(["jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"], 1)}

def normalize_join_key(v):
    """Same rule as the All Dashboard notebook: 19, 19.0 and "19" match; upper case."""
    if pd.isna(v):
        return ""
    s = str(v).strip()
    if s.endswith(".0") and s[:-2].replace("-", "").isdigit():
        s = s[:-2]
    return s.upper()

def date_from_name(name):
    """Last date in a file name -> datetime.date.
    Accepts 11Nov26 / 11 Nov 2026 and also month + year only (Oct26, Oct 2026 -> day 1)."""
    stem = os.path.splitext(os.path.basename(name))[0]
    hits = re.findall(r"(?:(\d{1,2})[ _\-.]?)?([A-Za-z]{3})[A-Za-z]*[ _\-.]?(\d{2}|\d{4})(?!\d)", stem)
    for d, m, y in reversed(hits):
        if m.lower() in MONTHS:
            y = int(y); y = y + 2000 if y < 100 else y
            try:
                return datetime.date(y, MONTHS[m.lower()], int(d) if d else 1)
            except ValueError:
                continue
    return None

def ym(y, m):            # running month number
    return y * 12 + (m - 1)

def parse_ym(s):
    y, m = str(s).split("-")
    return int(y), int(m)

def month_label(n):
    return f"{n // 12}-{n % 12 + 1:02d}"


In [ ]:
# ------------------------------------------------------------
# 1) Actual demand (D-1 ... D-12) from the latest pmovdcE
# ------------------------------------------------------------
if ACTUAL_D1_MONTH:
    y, m = parse_ym(ACTUAL_D1_MONTH); d1 = ym(y, m)
else:
    dt = date_from_name(ACTUAL_FILE)
    if dt is None:
        raise ValueError("Cannot read a date from ACTUAL_FILE's name -- set ACTUAL_D1_MONTH (e.g. '2026-10').")
    d1 = ym(dt.year, dt.month) - 1            # D-1 = the month before the pmovdcE date
print("D-1 month:", month_label(d1))

pm = pd.read_excel(ACTUAL_FILE, sheet_name=ACTUAL_SHEET, skiprows=ACTUAL_SKIPROWS)
pm.columns = [str(c).strip() for c in pm.columns]
need = ["Brc", "P/N"] + [f"D-{i}" for i in range(1, 13)]
missing = [c for c in need if c not in pm.columns]
if missing:
    raise ValueError(f"ACTUAL_FILE is missing columns: {missing}")
D_COLS = [f"D-{i}" for i in range(1, 13)]
pm = pm[need].copy()
pm["Brc"] = pm["Brc"].map(normalize_join_key)
pm["P/N"] = pm["P/N"].map(normalize_join_key)
pm[D_COLS] = pm[D_COLS].apply(pd.to_numeric, errors="coerce").fillna(0)

act_branch = pm.groupby(["Brc", "P/N"])[D_COLS].sum()                       # all agencies together
act_nat = pm.groupby("P/N")[D_COLS].sum()
act_nat.index = pd.MultiIndex.from_product([["NATIONAL"], act_nat.index])
actual = pd.concat([act_branch, act_nat])
actual.index.names = ["Brc", "P/N"]
print(f"Actual demand rows: {len(actual):,}  (branch rows + National rows)")


In [ ]:
# ------------------------------------------------------------
# 2) Forecast files -> F-k  (k = months back from D-1)
# ------------------------------------------------------------
files = list(FORECAST_FILES) or sorted(glob.glob(FORECAST_FILE_GLOB))
if not files:
    raise FileNotFoundError("No forecast files found. Fill FORECAST_FILES or FORECAST_FILE_GLOB.")

def read_forecast_file(path):
    """Returns a Series indexed by (Brc, P/N): the forecast value of that file."""
    parts = []
    for sh in FORECAST_SHEETS:
        try:
            df = pd.read_excel(path, sheet_name=sh, usecols=lambda c: str(c).strip().lower() in
                               {"brc", "agc", "p/n", FORECAST_VALUE_COLUMN.lower()})
        except ValueError:                      # sheet not in this file
            continue
        df.columns = [str(c).strip().lower() for c in df.columns]
        vcol = FORECAST_VALUE_COLUMN.lower()
        if "brc" not in df.columns:
            df["brc"] = "National"
        if "agc" not in df.columns:
            df["agc"] = "ALL AGC"
        df["brc"] = df["brc"].map(normalize_join_key)
        df["p/n"] = df["p/n"].map(normalize_join_key)
        df["agc"] = df["agc"].map(normalize_join_key)
        df[vcol] = pd.to_numeric(df[vcol], errors="coerce")
        parts.append(df[["brc", "agc", "p/n", vcol]].rename(columns={vcol: "f"}))
    df = pd.concat(parts, ignore_index=True)
    # Prefer the "ALL AGC" row of a Brc + P/N; if a Brc + P/N has none, add its agencies together.
    allagc = df[df["agc"] == "ALL AGC"].groupby(["brc", "p/n"])["f"].first()
    others = df[df["agc"] != "ALL AGC"].groupby(["brc", "p/n"])["f"].sum(min_count=1)
    s = allagc.combine_first(others)
    s.index.names = ["Brc", "P/N"]
    return s

hist = pd.DataFrame(index=pd.MultiIndex.from_arrays([[], []], names=["Brc", "P/N"]),
                    columns=[f"F-{i}" for i in range(1, 13)], dtype=float)
log = []
series_by_k = {}
for path in files:
    name = os.path.basename(path)
    if name in FORECAST_MONTH_OVERRIDE:
        y, m = parse_ym(FORECAST_MONTH_OVERRIDE[name]); fm = ym(y, m)
        src = "override"
    else:
        dt = date_from_name(name)
        if dt is None:
            log.append([name, None, None, None, 0, "SKIPPED - no date in the file name (use FORECAST_MONTH_OVERRIDE)"])
            continue
        fm = ym(dt.year, dt.month) + FORECAST_MONTH_OFFSET
        src = f"file date {dt:%d %b %Y}"
    k = d1 - fm + 1                     # k = 1 -> the forecast month is D-1
    if k < 1:
        log.append([name, src, month_label(fm), k, 0, "SKIPPED - no actual demand for that month yet"]); continue
    if k > 12:
        log.append([name, src, month_label(fm), k, 0, "SKIPPED - older than D-12"]); continue
    if k in series_by_k:
        log.append([name, src, month_label(fm), k, 0, f"SKIPPED - another file already covers D-{k}"]); continue
    s = read_forecast_file(path)
    series_by_k[k] = s
    log.append([name, src, month_label(fm), k, len(s), "USED"])
    print(f"D-{k:<2} {month_label(fm)}  {name}  ({len(s):,} rows)")

files_used = pd.DataFrame(log, columns=["File", "Month taken from", "Forecast month", "k (D-k)", "Rows", "Status"])
missing_k = [k for k in range(1, 13) if k not in series_by_k]
if missing_k:
    print("\nMonths WITHOUT a forecast file (F-k stays empty):", ", ".join(f"D-{k} ({month_label(d1 - k + 1)})" for k in missing_k))
files_used


In [ ]:
# ------------------------------------------------------------
# 3) Forecast error  E-k = D-k - F-k  and sigma
# ------------------------------------------------------------
# The latest pmovdcE is the template: one row per Brc + P/N (+ the National rows).
all_keys = actual.index
F = pd.DataFrame(index=all_keys, columns=[f"F-{i}" for i in range(1, 13)], dtype=float)
filled = {}
for k, s_k in series_by_k.items():
    col = s_k.reindex(all_keys)
    filled[k] = int(col.isna().sum())            # template rows not found in that month's file
    if MISSING_FORECAST.lower() == "zero":
        col = col.fillna(0.0)
    F[f"F-{k}"] = col
    extra = len(s_k.index.difference(all_keys))
    print(f"D-{k:<2} rows of the pmovdcE not in the forecast file: {filled[k]:,}"
          f"  ({'filled with 0' if MISSING_FORECAST.lower() == 'zero' else 'left blank'});"
          f"  forecast rows not in the pmovdcE (ignored): {extra:,}")
files_used["Template rows missing in file"] = files_used["k (D-k)"].map(filled)

A = actual.reindex(all_keys).fillna(0.0)          # a forecast row without any movement = no demand
A.columns = [f"D-{i}" for i in range(1, 13)]

a = A.to_numpy(float); f = F.to_numpy(float)
ok = ~np.isnan(f) & ~np.isnan(a)
err = np.where(ok, a - f, np.nan)
n = ok.sum(axis=1)

with np.errstate(invalid="ignore", divide="ignore"):
    mean_err = np.nanmean(np.where(ok, err, np.nan), axis=1)
    rmse = np.sqrt(np.nanmean(np.where(ok, err ** 2, np.nan), axis=1))
    mae = np.nanmean(np.where(ok, np.abs(err), np.nan), axis=1)
    std = np.full(len(n), np.nan)
    m2 = n >= 2
    dev = np.where(ok, err - mean_err[:, None], 0.0)
    std[m2] = np.sqrt((dev[m2] ** 2).sum(axis=1) / (n[m2] - 1))     # Excel STDEV (sample)
sigma = rmse if SIGMA_METHOD.upper() == "RMSE" else std
sigma = np.where(n >= max(SIGMA_MIN_MONTHS, 2), sigma, np.nan)

E = pd.DataFrame(err, index=all_keys, columns=[f"E-{i}" for i in range(1, 13)])
stats = pd.DataFrame({
    "Months used": n,
    "Mean Error (bias)": mean_err,
    "STDEV": std,
    "RMSE": rmse,
    "MAE": mae,
    "Sigma m": sigma,
    "Zero Months": (a <= 0).sum(axis=1),
    "Mean Actual": np.nanmean(np.where(ok, a, np.nan), axis=1),
    "Mean Forecast": np.nanmean(np.where(ok, f, np.nan), axis=1),
}, index=all_keys)
stats["CV"] = np.where(stats["Mean Forecast"] > 0, stats["Sigma m"] / stats["Mean Forecast"], np.nan)

print(f"Rows: {len(all_keys):,}   with Sigma m: {int(np.isfinite(sigma).sum()):,}   "
      f"(needs >= {SIGMA_MIN_MONTHS} months of actual + forecast)")
print("Sigma method:", SIGMA_METHOD.upper())
stats.describe().loc[["count", "mean", "50%", "max"]].round(2)


In [ ]:
# ------------------------------------------------------------
# 4) Export
# ------------------------------------------------------------
def flat(df):
    out = df.reset_index()
    out["Brc"] = out["Brc"].replace({"NATIONAL": "National"})
    return out

out_name = OUTPUT_FILE or f"Forecast History {month_label(d1)}.xlsx"
with pd.ExcelWriter(out_name, engine="openpyxl") as xw:
    flat(F).to_excel(xw, sheet_name="Forecast History", index=False)
    flat(pd.concat([E, stats], axis=1)).to_excel(xw, sheet_name="Forecast Error", index=False)
    flat(A).to_excel(xw, sheet_name="Actual", index=False)
    files_used.to_excel(xw, sheet_name="Files Used", index=False)
    notes = pd.DataFrame({"Notes": [
        "Forecast History (Brc, P/N, F-1..F-12) -> use as FORECAST_HISTORY_FILE in All_Dashboard_1_2_6_Claude, sheet 'Forecast History'.",
        "F-k = the forecast made for the month D-k (F-1 = most recent month, same order as D-1..D-12 in the pmovdcE file).",
        "E-k = Actual (D-k) - Forecast (F-k).  Bias = mean error; Sigma m = " + SIGMA_METHOD.upper() + " of the errors (needs >= " + str(SIGMA_MIN_MONTHS) + " months).",
        "Forecast value used: " + FORECAST_VALUE_COLUMN + ".  Brc = National is the company-wide row (used by FD Processed).",
        "Actual demand of a Brc + P/N adds all agencies together (same as the All Dashboard 'ALL AGC' rows).",
        "Template = the latest pmovdcE. A Brc + P/N missing from a forecast file: MISSING_FORECAST = " + MISSING_FORECAST + " ('zero' = forecast 0, error = actual; 'skip' = month ignored).",
        "D-1 month = " + month_label(d1) + ".",
    ]})
    notes.to_excel(xw, sheet_name="Notes", index=False)
    from openpyxl.styles import Font, PatternFill, Alignment
    for ws in xw.book.worksheets:
        for c in ws[1]:
            c.font = Font(bold=True, color="FFFFFF"); c.fill = PatternFill("solid", fgColor="1F4E78")
            c.alignment = Alignment(horizontal="center", wrap_text=True)
        ws.freeze_panes = "C2" if ws.title != "Notes" else "A2"
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = 12
    xw.book["Files Used"].column_dimensions["A"].width = 42
    xw.book["Files Used"].column_dimensions["F"].width = 55
    xw.book["Notes"].column_dimensions["A"].width = 140
print("Saved:", out_name)
